# Vertex Networking Scraper (v7 — skip category 1 & 2, start from 3)

Columns: `category, title, description, image, price, part_number, url`

This run is configured with:
```python
ONLY_THIS_CATEGORY = None
ALREADY_DONE_CATEGORIES = ["/category/1", "/category/2"]
```
So it will crawl **every category from 3 onward**, skipping 1 and 2 entirely.

**To scrape just ONE specific category instead**, set `ONLY_THIS_CATEGORY = "/category/5"` (for example) and it'll ignore `ALREADY_DONE_CATEGORIES` and only crawl that one.

All runs write to the same `vertexnetworking_products.csv` and skip product URLs already saved — safe to run this way with zero duplicates.

In [ ]:
!pip install requests beautifulsoup4

In [2]:
#!/usr/bin/env python3
"""
Scraper for vertexnetworking.co.uk
Walks every category -> subcategory -> product listing (with pagination),
visits each product page, and writes:
    category, title, description, image, price, part_number, url
to a CSV file INCREMENTALLY (row-by-row) so no progress is lost if it's
interrupted or crashes partway through.

Field notes (learned from inspecting a real product page):
- title: their on-page <h1> is truncated by their template, so the title
  is instead derived from the Description text (which contains the full,
  untruncated name before "Product Condition:").
- price: some products don't have a price at all and instead show a
  "Request A Quote" / "Get A Quote" button. For those, the price cell is
  written as "Get a Quote for price" instead of being left blank or $0.00.
- part_number: pulled from the "Part Number: XXXXX" line on the page.
- image: their real product photo may be lazy-loaded (data-src / srcset
  instead of a plain src), so several attributes are checked.

Requirements:
    pip install requests beautifulsoup4

Usage:
    python scrape_vertexnetworking.py

Output:
    vertexnetworking_products.csv   (written to as it goes)

If you stop the script and run it again later, it will skip any product
URLs already present in the CSV, so it resumes instead of starting over.
Delete the CSV first if you want a completely fresh run.
"""

import csv
import os
import time
import re
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

BASE_URL = "https://www.vertexnetworking.co.uk"
OUTPUT_CSV = "vertexnetworking_products.csv"
REQUEST_DELAY = 0.75   # seconds between requests, be polite to their server
TIMEOUT = 20
PROGRESS_EVERY = 25    # print a running total every N products scraped

# ---------------------------------------------------------------------------
# RUN ONE CATEGORY AT A TIME
# ---------------------------------------------------------------------------
# Set this to the category you want to scrape THIS run, e.g. "/category/2".
# Only listing pages whose URL contains this text will be crawled — every
# other category is skipped entirely (not even its pages are fetched).
#
# Set it to None to scrape every category NOT listed in ALREADY_DONE_CATEGORIES.
#
# All runs share the same CSV, and already-scraped product URLs are always
# skipped — so running one category per session never creates duplicates.
ONLY_THIS_CATEGORY = None

# Categories you've already fully finished in previous runs (used only when
# ONLY_THIS_CATEGORY is None, to skip them during a "run everything" pass).
ALREADY_DONE_CATEGORIES = ["/category/1", "/category/2"]

FIELDNAMES = ["category", "title", "description", "image", "price", "part_number", "url"]

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )
}

GENERIC_HEADINGS = {"vertex networking", "home", "products", "detail"}
QUOTE_TEXT_PATTERN = re.compile(r"request\s*a?\s*quote|get\s*a?\s*quote", re.I)

session = requests.Session()
session.headers.update(HEADERS)


def get_soup(url):
    """Fetch a URL and return a BeautifulSoup object, or None on failure."""
    try:
        resp = session.get(url, timeout=TIMEOUT)
        resp.raise_for_status()
        time.sleep(REQUEST_DELAY)
        return BeautifulSoup(resp.text, "html.parser")
    except requests.RequestException as e:
        print(f"  [warn] failed to fetch {url}: {e}")
        return None


def discover_category_links():
    """Find every /category/N and /subcategory/N link from the homepage nav."""
    soup = get_soup(BASE_URL)
    if soup is None:
        raise SystemExit("Could not load homepage — aborting.")

    links = set()
    for a in soup.find_all("a", href=True):
        href = a["href"]
        full = urljoin(BASE_URL, href)
        if re.search(r"/(category|subcategory)/\d+", full):
            links.add(full)
    return sorted(links)


def get_category_label(soup, fallback_url):
    """Try to pull a readable category/subcategory name from a listing page.
    Their site has a generic 'Vertex Networking' <h1> (site header/logo) that
    appears before the real category heading, so that generic one is skipped."""
    for h in soup.find_all(["h1", "h2"]):
        text = h.get_text(strip=True)
        if text and text.lower() not in GENERIC_HEADINGS and len(text) > 1:
            return text

    title = soup.find("title")
    if title and title.get_text(strip=True):
        page_title = title.get_text(strip=True).split("|")[0].strip()
        if page_title.lower() not in GENERIC_HEADINGS:
            return page_title

    return fallback_url


def find_product_links(soup):
    """Extract all product detail page links from a listing page."""
    links = set()
    for a in soup.find_all("a", href=True):
        href = urljoin(BASE_URL, a["href"])
        if "/product/" in href and href.endswith(".html"):
            links.add(href)
    return links


def find_next_page(soup, current_url):
    """Look for a 'next page' pagination link."""
    next_link = soup.find("a", rel="next")
    if next_link and next_link.get("href"):
        return urljoin(current_url, next_link["href"])

    for a in soup.find_all("a", href=True):
        text = a.get_text(strip=True).lower()
        if text in ("next", "»", "next »", ">"):
            return urljoin(current_url, a["href"])

    return None


def clean_description_text(text):
    """Strip repeated 'Description' / 'Product Description' header words that
    get scooped up when we grab the whole section's text."""
    text = re.sub(r"^\s*(Description\s*)+", "", text, flags=re.I)
    text = re.sub(r"^\s*(Product Description\s*)+", "", text, flags=re.I)
    return text.strip()


def is_nonzero_price(text):
    digits = re.sub(r"[^\d.]", "", text)
    try:
        return float(digits) > 0
    except (ValueError, TypeError):
        return False


def extract_description(soup):
    desc_section = soup.find(["div", "section"], id=re.compile(r"description", re.I))
    if not desc_section:
        desc_section = soup.find(
            ["div", "section"],
            class_=re.compile(r"(product-desc|description|details)", re.I),
        )
    if desc_section:
        text = clean_description_text(desc_section.get_text(" ", strip=True))
        if text:
            return text

    meta_desc = soup.find("meta", attrs={"name": "description"})
    if meta_desc and meta_desc.get("content"):
        return meta_desc["content"].strip()
    return None


def extract_title(soup, description):
    # Priority 1: derive from the description — it reliably starts with
    # "PARTNUMBER - Brand ... Product Name" followed by "Product Condition:",
    # and is more complete than their on-page <h1> (which gets truncated).
    if description:
        parts = re.split(r"Product Condition\s*:", description, flags=re.I)
        candidate = parts[0].strip(" -")
        if candidate and len(candidate) > 3:
            return candidate

    # Priority 2: a real <h1>/<h2> that isn't the generic site header.
    for h in soup.find_all(["h1", "h2"]):
        text = h.get_text(strip=True)
        if text and text.lower() not in GENERIC_HEADINGS and len(text) > 3:
            return text

    # Priority 3: meta/og:title as an absolute last resort.
    og_title = soup.find("meta", property="og:title")
    if og_title and og_title.get("content"):
        return og_title["content"].strip()
    return None


def extract_image(soup):
    """Their real product photo may be lazy-loaded, so check several
    possible attributes, not just src. Real product images tend to come
    LATER in the page than banner/logo images, so prefer the last match."""
    candidates = []
    for img in soup.find_all("img"):
        for attr in ("src", "data-src", "data-original", "data-lazy-src", "data-image"):
            val = img.get(attr)
            if val and "/product_images/" in val:
                candidates.append(urljoin(BASE_URL, val))
                break
        else:
            srcset = img.get("srcset")
            if srcset and "/product_images/" in srcset:
                first_url = srcset.split(",")[0].strip().split(" ")[0]
                candidates.append(urljoin(BASE_URL, first_url))

    if candidates:
        return candidates[-1]

    og_image = soup.find("meta", property="og:image")
    if og_image and og_image.get("content"):
        return urljoin(BASE_URL, og_image["content"].strip())
    return None


def extract_price(soup):
    # 1. class="...price..." element with a non-zero value
    for tag in soup.find_all(["span", "div", "p"], class_=re.compile(r"price", re.I)):
        m = re.search(r"[\$£€]\s?[\d,]+\.?\d*", tag.get_text(" ", strip=True))
        if m and is_nonzero_price(m.group(0)):
            return m.group(0).strip()

    # 2. price-looking data embedded in <script> tags
    for script in soup.find_all("script"):
        if not script.string:
            continue
        m = re.search(r'"price"\s*:\s*"?\$?£?€?\s?([\d]+\.?\d*)"?', script.string)
        if m and is_nonzero_price(m.group(1)):
            return f"${m.group(1)}"

    # 3. any non-zero currency-looking value anywhere on the page
    #    (skips things like the header's "Shopping Cart: $0.00")
    for m in re.finditer(r"[\$£€]\s?[\d,]+\.\d{2}", soup.get_text(" ", strip=True)):
        if is_nonzero_price(m.group(0)):
            return m.group(0).strip()

    # 4. No price found at all — check if this is a "request a quote" item
    if soup.find(string=QUOTE_TEXT_PATTERN):
        return "Get a Quote for price"

    return None


def extract_part_number(soup):
    text = soup.get_text(" ", strip=True)
    m = re.search(r"Part\s*Number\s*:?\s*([A-Za-z0-9][A-Za-z0-9\-\/\.]*)", text, re.I)
    if m:
        return m.group(1).strip()
    return None


def scrape_product(url, category_label):
    """Visit a product page and extract all fields."""
    soup = get_soup(url)
    if soup is None:
        return None

    description = extract_description(soup)
    title = extract_title(soup, description)
    image = extract_image(soup)
    price = extract_price(soup)
    part_number = extract_part_number(soup)

    return {
        "category": category_label,
        "title": title or "",
        "description": description or "",
        "image": image or "",
        "price": price or "",
        "part_number": part_number or "",
        "url": url,
    }


def load_already_scraped():
    """If the CSV already exists (from a previous run), load which product
    URLs are already done so we can skip them and resume."""
    done = set()
    if os.path.exists(OUTPUT_CSV):
        with open(OUTPUT_CSV, "r", newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                if row.get("url"):
                    done.add(row["url"])
    return done


def open_csv_writer():
    """Open the CSV in append mode, writing the header only if new."""
    file_exists = os.path.exists(OUTPUT_CSV)
    f = open(OUTPUT_CSV, "a", newline="", encoding="utf-8")
    writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
    if not file_exists:
        writer.writeheader()
        f.flush()
    return f, writer


def main():
    already_done = load_already_scraped()
    if already_done:
        print(f"Resuming — {len(already_done)} products already in {OUTPUT_CSV}, will skip those.")

    csv_file, writer = open_csv_writer()
    scraped_count = len(already_done)

    try:
        print("Discovering category/subcategory links...")
        category_links = discover_category_links()
        print(f"Found {len(category_links)} category/subcategory links total.")

        if ONLY_THIS_CATEGORY:
            category_links = [c for c in category_links if ONLY_THIS_CATEGORY in c]
            print(f"ONLY_THIS_CATEGORY = '{ONLY_THIS_CATEGORY}' — "
                  f"{len(category_links)} matching link(s) will be crawled.")
        else:
            category_links = [
                c for c in category_links
                if not any(done in c for done in ALREADY_DONE_CATEGORIES)
            ]
            print(f"Skipping {ALREADY_DONE_CATEGORIES} — "
                  f"{len(category_links)} category links remain.")

        if not category_links:
            print("No matching category links found — check ONLY_THIS_CATEGORY "
                  "matches a real category URL (e.g. run once with it set to "
                  "None to see the full list printed above).")
            return

        for cat_index, cat_url in enumerate(category_links, start=1):
            print(f"\n[{cat_index}/{len(category_links)}] Category page: {cat_url}")
            page_url = cat_url
            visited_pages = set()

            while page_url and page_url not in visited_pages:
                visited_pages.add(page_url)
                soup = get_soup(page_url)
                if soup is None:
                    break

                label = get_category_label(soup, cat_url)
                product_links = find_product_links(soup)
                new_links = [p for p in product_links if p not in already_done]
                print(f"  {page_url} -> {len(product_links)} product links "
                      f"({len(new_links)} new)")

                for p_url in new_links:
                    row = scrape_product(p_url, label)
                    already_done.add(p_url)
                    if row:
                        writer.writerow(row)
                        csv_file.flush()  # write to disk immediately
                        scraped_count += 1
                        if scraped_count % PROGRESS_EVERY == 0:
                            print(f"  >>> progress: {scraped_count} products saved so far")

                page_url = find_next_page(soup, page_url)

    except KeyboardInterrupt:
        print("\n[stopped by user] Progress so far is safely saved in the CSV.")
    finally:
        csv_file.close()

    print(f"\nTotal products in {OUTPUT_CSV}: {scraped_count}")
    print("Done (or safely stopped — rerun the script anytime to resume).")


if __name__ == "__main__":
    main()


Resuming — 62857 products already in vertexnetworking_products.csv, will skip those.
Discovering category/subcategory links...
Found 120 category/subcategory links total.
Skipping ['/category/1', '/category/2'] — 111 category links remain.

[1/111] Category page: https://www.vertexnetworking.co.uk/category/3
  https://www.vertexnetworking.co.uk/category/3 -> 20 product links (0 new)
  https://www.vertexnetworking.co.uk/category/3?page=2 -> 20 product links (17 new)
  https://www.vertexnetworking.co.uk/category/3?page=3 -> 20 product links (20 new)
  >>> progress: 62875 products saved so far
  https://www.vertexnetworking.co.uk/category/3?page=4 -> 20 product links (20 new)
  >>> progress: 62900 products saved so far
  https://www.vertexnetworking.co.uk/category/3?page=5 -> 20 product links (20 new)
  >>> progress: 62925 products saved so far
  https://www.vertexnetworking.co.uk/category/3?page=6 -> 20 product links (20 new)
  >>> progress: 62950 products saved so far
  https://www.vert